### Parquet

Apache Parquet is an open source, column-oriented data file format designed for efficient data storage and retrieval.

It provides columnar compression, saves storage space and allows for reading individual columns instead of entire files.

It is a file format that works exceptionally well with Apache Spark and is in fact the default file format. 

It supports complex types. This means that if your column is an array/struct/map, will be able to read and write.


It is similar to other columnar-storage file formats available in Hadoop, namely RCFile and ORC.


* **benifits:**

  - Amazon Athena and Spectrum will charge based on the amount of data scanned per query.
  - Google and Amazon will charge you according to the amount of data stored on GS/S3.


[https://parquet.apache.org/docs/file-format/](https://parquet.apache.org/docs/file-format/)

In [9]:
from pyspark.sql.session import SparkSession

spark  = (

    SparkSession
    .builder
    .getOrCreate()
    
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/12 15:41:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [15]:
! ls -lrth ../data/flight-data/parquet/

total 4.0K
drwxrwxr-x 2 suanki suanki 4.0K Oct 16 22:28 2010-summary.parquet


In [17]:
df = spark.read.format("parquet").load("../data/flight-data/parquet/2010-summary.parquet")
df.show(5)


+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


Columner storage technique:

<img src="../data/pictures/parquet_understanding.png" width="500">



Parquet uses the hybrid approach of sequentially storing chunks of columns.


<img src="../data/pictures/parquet_hybrid_approach.png" width="500">


Advantages of Columnar Storage Over Row-Based Storage:

1. compression efficiency: column aften has same data type, so, it apply columner-compression
2. column prunning : while query ,irrelevant column can be skipped based on metadata
3. aggregation performance: when we aggregate we usually do it on selected columns
4. predicate pushdown : can apply filter early in qeury execution by analyzing metadata
5. Schema Evaluation : schema can be modified or new column can be added without disrupting existing data.
                       using 'mergeSchema'

6. Metadata: Parquet files contain metadata that describes the schema and encoding of the data, enhancing query optimization.

## Parquet’s Hybrid Approach

Parquet combines row-oriented grouping with column-oriented storage. It divides a file into **row groups** (horizontal partitions of rows), then stores each column’s data together within each row group.

### File structure

- **File:** A Parquet file contains the data and metadata that describes it, including its schema and row groups.
- **Row group:** A logical horizontal partition of the dataset. Each row group contains one **column chunk** for each column.
- **Column chunk:** The data for a particular column within a row group. Column chunks are stored contiguously in the file.
- **Page:** A column chunk is divided into pages, which are the units of encoding and compression. Different page types may appear within a column chunk.

> **HDFS block:** An HDFS block is a storage-layer concept, not a component of the Parquet file format. Parquet files can be stored on HDFS as well as other storage systems.



<img src="../data/pictures/parquet_internal.png" width="700">

# Writing Parquet Files in Python

Python provides several libraries that enable you to write data into Parquet files. One popular library is pyarrow

In [24]:
import os
import pandas as pd

data = {'Name': ['Alice', 'Bob', 'Charlie'], 'Age': [25, 30, 28]}
df = pd.DataFrame(data)

output_path = "../data/tmp/pyarrow/sample.parquet"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Directly write the pandas DataFrame to Parquet using the pyarrow engine
df.to_parquet(output_path, engine='pyarrow')